# 06 — Failure analysis

Per `plan.md` §10 entry for `06_failure_analysis.ipynb`.

* Top false positives gallery
* Missed-hole gallery (top false negatives)
* Score vs (`axial_ratio`, `diameter_pc`, `vexp_kms`, `hole_type`)
  scatter to find systematic blind spots.

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from hishells.augment import no_augment
from hishells.catalog import LOGO_GALAXIES_19, load_catalog
from hishells.cubes import sigma_rms
from hishells.data import (
    CubeStore,
    DatasetConfig,
    LOGOSplitter,
    ShellWindowDataset,
    make_subset,
)
from hishells.model import build_model
from hishells.predict import predict_dataset_mc
from hishells.train import load_checkpoint
from hishells.windows import NegSampleConfig, build_window_table

In [ ]:
REPO = Path('..').resolve()
ABLATION = 'v1_baseline'
FOLD = 'NGC_2403'  # change me

cat = load_catalog(REPO / 'Data' / 'J_AJ_141_23')
holes = cat.filter(hole_types=(2, 3), downloaded_in=REPO / 'Data' / 'THINGS')
cubes = CubeStore(REPO / 'Data' / 'THINGS', max_cubes=2)
sigmas = {g: sigma_rms(cubes(g)) for g in LOGO_GALAXIES_19 if g in set(holes['galaxy_id'])}
neg_cfg = NegSampleConfig(ratio=5.0, hard_frac=0.75, rng_seed=0)
table = build_window_table(holes, {g: cubes(g) for g in sigmas}, neg_cfg, cube_sigmas=sigmas)
splitter = LOGOSplitter(table, galaxies=tuple(sigmas.keys()), val_frac=0.10, rng_seed=0)
fold = next(f for f in splitter if f.test_galaxy == FOLD)
ds = ShellWindowDataset(table, cubes, sigma_rms_by_galaxy=sigmas, config=DatasetConfig(window_pix=96, augment=no_augment()))
test_set = make_subset(ds, fold.test_idx)

model = build_model('small')
load_checkpoint(REPO / 'results' / 'checkpoints' / ABLATION / f'{FOLD}.pt', model)
mc, test_labels, _ = predict_dataset_mc(model, test_set, T=50, batch_size=64)
rows = table.iloc[fold.test_idx].reset_index(drop=True).copy()
rows['score'] = mc.score_mean
rows['score_std'] = mc.score_std
rows.head()

## Top false-positive and false-negative galleries

Operating point comes from the corresponding `results/ablations.csv`
row's `threshold` column.

In [ ]:
abl = pd.read_csv(REPO / 'results' / 'ablations.csv')
tau = float(abl[(abl['name'] == ABLATION) & (abl['fold'] == FOLD)]['threshold'].iloc[0])
print(f'tau = {tau:.3f}')

rows['pred'] = (rows['score'] >= tau).astype(int)
fp = rows[(rows['label'] == 0) & (rows['pred'] == 1)].sort_values('score', ascending=False).head(8)
fn = rows[(rows['label'] == 1) & (rows['pred'] == 0)].sort_values('score').head(8)

from hishells.pvcut import extract_window_for_hole
from hishells.windows import normalize_window

def gallery(rows_subset, title):
    fig, axes = plt.subplots(1, len(rows_subset), figsize=(2 * len(rows_subset), 2.5))
    if len(rows_subset) == 0: return
    for ax, (_, r) in zip(np.atleast_1d(axes), rows_subset.iterrows()):
        cube = cubes(str(r['galaxy_id']))
        win = normalize_window(extract_window_for_hole(cube, r.to_dict(), window_pix=96), sigmas[str(r['galaxy_id'])])
        ax.imshow(win, origin='lower', cmap='magma', vmin=-2, vmax=8)
        ax.set_title(f's={r.score:.2f} ± {r.score_std:.2f}', fontsize=7)
        ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(title)

gallery(fp, f'top false positives @ tau={tau:.2f}')
gallery(fn, f'top missed positives @ tau={tau:.2f}')

## Score vs B11 properties

Look for systematic patterns: do small holes / low-Vexp / type-2 holes
score systematically lower? That's a signal to revisit §2.4 / §2.1.

In [ ]:
pos = rows[rows['label'] == 1]
fig, axes = plt.subplots(1, 4, figsize=(15, 3.5))
for ax, col in zip(axes, ['axial_ratio', 'diameter_pc', 'vexp_kms', 'hole_type']):
    if col not in pos.columns:
        ax.set_visible(False); continue
    ax.scatter(pos[col], pos['score'], s=14, alpha=0.6)
    ax.set_xlabel(col); ax.set_ylabel('score')
fig.tight_layout()